# IoTShield — 10: Ek deneyler
**SE3513 · WP-EK · ASEADOS-SDN-IoT** · Plan: "IoTShield — Ek Deneyler Planı" (Berat, 28 Kasım 2026)

Hocaların "bunu denediniz mi?" sorularına ölçülmüş sonuçla cevap vermek için. Kurallar:
- Resmî CSV'lere ve `src/`'ye dokunulmaz; sonuçlar sadece `reports/results_extra.csv`'ye yazılır.
- Katlar `get_folds()`'tan; mümkün olan her deney `run_cv()` ile kaydedilir.
- Her satırın `note` sütununda deney numarası ve esnetilen kural yazar (ör. `EK-11 ...; Kural 5 dışı`).
- Kesin dil yok (Kural 8).

## 0. Kurulum

In [1]:
import os
import sys
sys.path.append('..')

import numpy as np
import pandas as pd

from src.config import DATA_PATH, RANDOM_STATE, F_TIMING, F_FULL, FEATS
from src.preprocessing import clean
from src.cv import (get_folds, print_fold_table, run_cv, compute_metrics, drop_conflicts,
                    subset_macro_f1, append_row, assign_sessions_to_folds, make_groups,
                    CLASSES, N_SPLITS, SESSION_GAP_MINUTES)
from imblearn.pipeline import Pipeline
from lightgbm import LGBMClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef

pd.set_option('display.max_columns', 30)
EXTRA_CSV = '../reports/results_extra.csv'
SHOW_COLS = ['experiment', 'model', 'feature_set', 'macro_f1_mean', 'macro_f1_std', 'macro_f1_clean',
             'macro_f1_broken', 'recall_Bot', 'bot_ddos_confusion', 'note']

## 1. Veri ve katlar
`clean()` verisi; katlar bir kez alınır, bütün deneylerde aynı katlar. Sonuç dosyası her çalıştırmada sıfırdan yazılır (`run_cv` satır ekler; tekrar çalıştırmada çift satır olmasın).

In [2]:
data = clean(pd.read_csv(DATA_PATH, low_memory=False))
print('veri:', data.shape)

if os.path.exists(EXTRA_CSV):
    os.remove(EXTRA_CSV)

folds = get_folds(data)
fold_table = print_fold_table(data, folds)
results = {}   # deney kodu -> (row, confusion matrix, OOF tahminleri)

clean: dropped 70 rows with inf/NaN
clean: clipped negative values to 0 in 427 rows


clean: removed 29930 duplicate rows (features + label)
clean: final shape (427044, 63)
veri: (427044, 63)


Test kısmındaki satır sayıları (sınıf × kat):
        fold_0  fold_1  fold_2
Label                         
Benign   76910   79965   76872
Bot       1757    2585    4615
DDoS     23953   10599   15717
Dos      16482   16711   93313
Prob      2803    2316    2446
Test kısmındaki oturum sayıları (sınıf × kat):
        fold_0  fold_1  fold_2
Label                         
Benign       9       3      11
Bot          1       1       1
DDoS         1       1       2
Dos         19       3       1
Prob         1       1       4


## 2. Referans: E0 ve Set B
Sonraki deneylerin karşılaştırma noktası ve `run_cv`'nin tekrarlanabilirlik kontrolü. Beklenen (resmî CSV'lerden): E0 (F_TIMING) **0,418 ± 0,119**, bozuk 0,324; Set B (F_FULL) **0,499 ± 0,052**, bozuk 0,339. OOF tahminleri deney 14'te kullanılır.

In [3]:
EXPECTED_E0 = 0.418      # results_imbalance.csv, E0 grouped
EXPECTED_SET_B = 0.499   # results_features.csv, ablation_B grouped


def make_lgbm(seed=RANDOM_STATE):
    """Sabit deney modeli (Kural 5): LightGBM, varsayılan parametreler; sadece tohum değişebilir."""
    return Pipeline([('model', LGBMClassifier(random_state=seed, verbose=-1))])


results['E0'] = run_cv(make_lgbm(), data, F_TIMING, EXTRA_CSV, experiment='EK-ref-E0', sampler='none',
                       feature_set='F_TIMING', folds=folds, note='reference (same as E0)')
results['SetB'] = run_cv(make_lgbm(), data, F_FULL, EXTRA_CSV, experiment='EK-ref-SetB', sampler='none',
                         feature_set='F_FULL', folds=folds, note='reference (same as Set B)')

assert round(results['E0'][0]['macro_f1_mean'], 3) == EXPECTED_E0, results['E0'][0]['macro_f1_mean']
assert round(results['SetB'][0]['macro_f1_mean'], 3) == EXPECTED_SET_B, results['SetB'][0]['macro_f1_mean']
print('Referanslar resmî sonuçlarla aynı.')

Kat 0


  eğitim: 301069 satır (çelişki atılan: 4070 ) | test: 121905 | fit: 2.6 sn
Kat 1


  eğitim: 309862 satır (çelişki atılan: 5006 ) | test: 112176 | fit: 2.3 sn
Kat 2


  eğitim: 230078 satır (çelişki atılan: 4003 ) | test: 192963 | fit: 2.2 sn


Kat F1'leri: [0.4262 0.5334 0.2952] → 0.4183 ± 0.1193
CSV'ye eklendi: ../reports/results_extra.csv


Kat 0


  eğitim: 304139 satır (çelişki atılan: 1000 ) | test: 121905 | fit: 2.9 sn
Kat 1


  eğitim: 314008 satır (çelişki atılan: 860 ) | test: 112176 | fit: 3.2 sn
Kat 2


  eğitim: 233464 satır (çelişki atılan: 617 ) | test: 192963 | fit: 6.5 sn


Kat F1'leri: [0.4554 0.5563 0.4847] → 0.4988 ± 0.0519
CSV'ye eklendi: ../reports/results_extra.csv


Referanslar resmî sonuçlarla aynı.


## 3. Deney 11: Şans seviyesi
Soru: Model gerçekten bir şey öğreniyor mu? E0'ın bozuk F1'i (0,324) neye göre düşük?
- **(a) DummyClassifier:** `most_frequent` (hep Benign) ve `stratified` (sınıf oranlarına göre rastgele). `run_cv` ile.
- **(b) Karıştırılmış etiket:** Her eğitim katında etiketler rastgele karıştırılır, LightGBM bununla eğitilir, gerçek test etiketleriyle ölçülür. 3 tekrar.
  `run_cv` eğitim etiketini değiştiremediği için döngü burada yazılır: katlar `get_folds()`'tan, çelişkiler gerçek etiketlerle `drop_conflicts` ile atılır, metrikler `compute_metrics` / `subset_macro_f1` ile (plan "Kurallar").

In [4]:
DUMMY_STRATEGIES = ['most_frequent', 'stratified']

for strategy in DUMMY_STRATEGIES:
    print('=== EK-11a', strategy, '===')
    pipe = Pipeline([('model', DummyClassifier(strategy=strategy, random_state=RANDOM_STATE))])
    results['11a_' + strategy] = run_cv(pipe, data, F_TIMING, EXTRA_CSV, experiment='EK-11a-' + strategy,
                                        sampler='none', feature_set='F_TIMING', folds=folds,
                                        note='EK-11 chance level (Dummy ' + strategy + '); Kural 5 dışı')

=== EK-11a most_frequent ===
Kat 0


  eğitim: 301069 satır (çelişki atılan: 4070 ) | test: 121905 | fit: 0.1 sn
Kat 1


  eğitim: 309862 satır (çelişki atılan: 5006 ) | test: 112176 | fit: 0.1 sn
Kat 2


  eğitim: 230078 satır (çelişki atılan: 4003 ) | test: 192963 | fit: 0.1 sn


Kat F1'leri: [0.1547 0.1665 0.114 ] → 0.1451 ± 0.0276
CSV'ye eklendi: ../reports/results_extra.csv


=== EK-11a stratified ===
Kat 0


  eğitim: 301069 satır (çelişki atılan: 4070 ) | test: 121905 | fit: 0.1 sn
Kat 1


  eğitim: 309862 satır (çelişki atılan: 5006 ) | test: 112176 | fit: 0.1 sn
Kat 2


  eğitim: 230078 satır (çelişki atılan: 4003 ) | test: 192963 | fit: 0.1 sn


Kat F1'leri: [0.1835 0.1863 0.1707] → 0.1801 ± 0.0083
CSV'ye eklendi: ../reports/results_extra.csv


In [5]:
N_SHUFFLE_REPEATS = 3


def own_loop_row(experiment, feature_set, fold_f1s, oof, note):
    """run_cv kullanmayan deneyler için run_cv ile aynı sütun sırasında bir sonuç satırı kurar."""
    y_true = data['Label'].to_numpy()
    metrics = compute_metrics(y_true, oof, data['broken'])
    row = {'experiment': experiment, 'model': 'LGBMClassifier', 'sampler': 'none',
           'feature_set': feature_set, 'cv': 'grouped',
           'macro_f1_mean': np.mean(fold_f1s), 'macro_f1_std': np.std(fold_f1s, ddof=1),
           'macro_f1_clean': metrics['macro_f1_clean'], 'macro_f1_broken': metrics['macro_f1_broken']}
    for class_name in CLASSES:
        row['recall_' + class_name] = metrics['recall_' + class_name]
    row['bot_ddos_confusion'] = metrics['bot_ddos_confusion']
    row['fit_time_s'] = np.nan
    row['pred_ms_per_flow'] = np.nan
    row['note'] = note
    return row


def run_shuffled_labels(feats, repeat):
    """Her eğitim katında etiketleri karıştırıp LightGBM eğitir, gerçek test etiketleriyle ölçer.

    Satırı EXTRA_CSV'ye ekler ve satır sözlüğünü döndürür.
    """
    rng = np.random.default_rng(RANDOM_STATE + repeat)
    y_true = data['Label'].to_numpy()
    oof = np.empty(len(data), dtype=object)
    fold_f1s = []
    for train_idx, test_idx in folds:
        # Çelişkiler gerçek etiketlerle atılır (run_cv ile aynı eğitim satırları), sonra etiket karıştırılır.
        train = drop_conflicts(data.iloc[train_idx], feats)
        shuffled_labels = rng.permutation(train['Label'].to_numpy())
        model = LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)
        model.fit(train[feats], shuffled_labels)
        predictions = model.predict(data.iloc[test_idx][feats])
        oof[test_idx] = predictions
        fold_f1s.append(subset_macro_f1(y_true[test_idx], predictions, CLASSES))
    row = own_loop_row('EK-11b-shuffle' + str(repeat), 'F_TIMING', fold_f1s, oof.astype(y_true.dtype),
                       'EK-11 chance level (shuffled train labels, repeat ' + str(repeat) + '); own loop, not run_cv')
    print('EK-11b tekrar', repeat, '| kat F1:', np.round(fold_f1s, 4), '| bozuk F1:', round(row['macro_f1_broken'], 4))
    append_row(row, EXTRA_CSV)
    return row


for repeat in range(N_SHUFFLE_REPEATS):
    results['11b_' + str(repeat)] = (run_shuffled_labels(F_TIMING, repeat), None, None)

chance_rows = [results['E0'][0]]
for key in results:
    if key.startswith('11'):
        chance_rows.append(results[key][0])
display(pd.DataFrame(chance_rows)[SHOW_COLS].round(3))

EK-11b tekrar 0 | kat F1: [0.2536 0.1672 0.1135] | bozuk F1: 0.0989
CSV'ye eklendi: ../reports/results_extra.csv


EK-11b tekrar 1 | kat F1: [0.2683 0.1666 0.1162] | bozuk F1: 0.0977
CSV'ye eklendi: ../reports/results_extra.csv


EK-11b tekrar 2 | kat F1: [0.1559 0.167  0.1139] | bozuk F1: 0.0996
CSV'ye eklendi: ../reports/results_extra.csv


,experiment,model,feature_set,macro_f1_mean,macro_f1_std,macro_f1_clean,macro_f1_broken,recall_Bot,bot_ddos_confusion,note
0,EK-ref-E0,LGBMClassifier,F_TIMING,0.418,0.119,0.512,0.324,0.031,0.140,reference (same as E0)
1,EK-11a-most_frequent,DummyClassifier,F_TIMING,0.145,0.028,0.388,0.101,0.000,0.000,EK-11 chance level (Dummy most_frequent); Kura...
2,EK-11a-stratified,DummyClassifier,F_TIMING,0.180,0.008,0.410,0.150,0.019,0.038,EK-11 chance level (Dummy stratified); Kural 5...
3,EK-11b-shuffle0,LGBMClassifier,F_TIMING,0.178,0.071,0.441,0.099,0.000,0.000,"EK-11 chance level (shuffled train labels, rep..."
4,EK-11b-shuffle1,LGBMClassifier,F_TIMING,0.184,0.077,0.457,0.098,0.000,0.000,"EK-11 chance level (shuffled train labels, rep..."
5,EK-11b-shuffle2,LGBMClassifier,F_TIMING,0.146,0.028,0.389,0.100,0.000,0.000,"EK-11 chance level (shuffled train labels, rep..."


**Bulgu (11):**
- Şans seviyesi (F_TIMING, aynı katlar): Dummy `most_frequent` 0,145 ± 0,028 (bozuk 0,101), `stratified` 0,180 ± 0,008 (bozuk 0,150), karıştırılmış etiketle LightGBM 0,146–0,184 (bozuk 0,098–0,100, 3 tekrar).
- E0: 0,418 ± 0,119, bozuk **0,324**. Bozuk alt kümede E0, karıştırılmış etiket seviyesinin (~0,10) yaklaşık 3 katı; yani model bozuk satırlarda da etiketle ilgili bir şey öğreniyor.
- Bot recall'u şans modellerinde 0,000–0,019, E0'da 0,031: Bot'ta E0 şans seviyesine yakın.

## 4. Deney 14: Başka metrikler
Soru: Neden macro F1? Yeni model eğitilmez; E0 ve Set B'nin OOF tahminlerinden (Bölüm 2) ek metrikler hesaplanır.
Tüm / sağlam / bozuk ayrı ayrı. Macro F1, `run_cv` ile aynı olsun diye `subset_macro_f1` ile (alt kümede olmayan sınıf ortalamaya girmez). Tablo CSV'ye yazılmaz (plan).

In [6]:
def extra_metrics(y_true, y_pred):
    """Bir alt küme için accuracy, balanced accuracy, weighted F1, macro F1 ve MCC sözlüğü döndürür."""
    return {'accuracy': accuracy_score(y_true, y_pred),
            'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
            'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
            'macro_f1': subset_macro_f1(y_true, y_pred, CLASSES),
            'mcc': matthews_corrcoef(y_true, y_pred)}


y_true_all = data['Label'].to_numpy()
broken_mask = data['broken'].to_numpy()
subsets = {'tüm': np.ones(len(data), dtype=bool), 'sağlam': ~broken_mask, 'bozuk': broken_mask}

metric_rows = []
for config in ['E0', 'SetB']:
    oof = results[config][2].to_numpy()
    for subset_name, mask in subsets.items():
        row = {'config': config, 'alt küme': subset_name}
        row.update(extra_metrics(y_true_all[mask], oof[mask]))
        metric_rows.append(row)

metrics_table = pd.DataFrame(metric_rows)
display(metrics_table.round(3))

/home/dogukandireksiz/Desktop/projects/mlproject/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/home/dogukandireksiz/Desktop/projects/mlproject/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,config,alt küme,accuracy,balanced_accuracy,weighted_f1,macro_f1,mcc
0,E0,tüm,0.623,0.410,0.552,0.388,0.350
1,E0,sağlam,0.675,0.560,0.588,0.512,0.263
2,E0,bozuk,0.497,0.321,0.459,0.324,0.233
3,SetB,tüm,0.757,0.511,0.745,0.520,0.591
4,SetB,sağlam,0.863,0.814,0.854,0.837,0.716
5,SetB,bozuk,0.499,0.335,0.466,0.339,0.241


**Bulgu (14):**
- Accuracy her alt kümede macro F1'den yüksek (ör. Set B tüm: accuracy 0,757, macro F1 0,520). Fark Benign'in çoğunluk olmasından: accuracy Benign'i doğru bilmekle yükseliyor, macro F1 her sınıfa eşit ağırlık veriyor. Macro F1'i bu yüzden seçtik.
- Bozuk alt kümede bütün metrikler düşük ve E0 ile Set B neredeyse aynı: accuracy 0,497 / 0,499, balanced accuracy 0,321 / 0,335, MCC 0,233 / 0,241. Metrik değiştirmek bozuk alt kümedeki tabloyu değiştirmiyor.
- Not: Buradaki "tüm" macro F1 (E0 0,388, Set B 0,520) birleşik OOF tahminlerinden; resmî tablodaki 0,418 / 0,499 kat ortalaması. İkisi farklı hesap, karıştırılmamalı. Sağlam ve bozuk F1 resmî sayılarla aynı (0,512 / 0,324, 0,837 / 0,339).

## 5. Deney 6: Model tohumu kararlılığı
Soru: E0 ile E9 / Set B arasındaki küçük farklar gerçek mi, yoksa tohum oynaması mı? E0 ve Set B, 5 farklı `random_state` ile; katlar sabit.
(E9 bu notebook'ta henüz yok; E9'un tohum satırları Berat'ın kısmında eklenecek.)

In [7]:
SEEDS = [0, 1, 7, 42, 123]
SEED_CONFIGS = {'E0': (F_TIMING, 'F_TIMING'), 'SetB': (F_FULL, 'F_FULL')}

seed_rows = []
for config, (feats, feature_set_name) in SEED_CONFIGS.items():
    for seed in SEEDS:
        print('=== EK-6', config, 'seed', seed, '===')
        row, cm, oof = run_cv(make_lgbm(seed), data, feats, EXTRA_CSV, experiment='EK-6-' + config + '-seed' + str(seed),
                              sampler='none', feature_set=feature_set_name, folds=folds,
                              note='EK-6 seed stability, random_state=' + str(seed))
        row['config'] = config
        row['seed'] = seed
        seed_rows.append(row)

seed_table = pd.DataFrame(seed_rows)
display(seed_table[['config', 'seed', 'macro_f1_mean', 'macro_f1_std', 'macro_f1_broken']].round(4))

seed_summary = seed_table.groupby('config')[['macro_f1_mean', 'macro_f1_broken']].agg(['min', 'max'])
display(seed_summary.round(4))

=== EK-6 E0 seed 0 ===
Kat 0


  eğitim: 301069 satır (çelişki atılan: 4070 ) | test: 121905 | fit: 2.3 sn
Kat 1


  eğitim: 309862 satır (çelişki atılan: 5006 ) | test: 112176 | fit: 3.0 sn
Kat 2


  eğitim: 230078 satır (çelişki atılan: 4003 ) | test: 192963 | fit: 2.8 sn


Kat F1'leri: [0.4302 0.536  0.2924] → 0.4195 ± 0.1221
CSV'ye eklendi: ../reports/results_extra.csv


=== EK-6 E0 seed 1 ===
Kat 0


... (42 outputs left out)

  eğitim: 233464 satır (çelişki atılan: 617 ) | test: 192963 | fit: 2.9 sn


Kat F1'leri: [0.4638 0.5537 0.4904] → 0.5026 ± 0.0462
CSV'ye eklendi: ../reports/results_extra.csv


,config,seed,macro_f1_mean,macro_f1_std,macro_f1_broken
0,E0,0,0.4195,0.1221,0.3216
1,E0,1,0.4244,0.1194,0.3248
2,E0,7,0.4210,0.1202,0.3230
3,E0,42,0.4183,0.1193,0.3240
4,E0,123,0.4222,0.1255,0.3232
5,SetB,0,0.5071,0.0469,0.3430
6,SetB,1,0.5018,0.0473,0.3415
7,SetB,7,0.5019,0.0456,0.3417
8,SetB,42,0.4988,0.0519,0.3392
9,SetB,123,0.5026,0.0462,0.3408


macro_f1_mean         macro_f1_broken        
                 min     max             min     max
config                                              
E0            0.4183  0.4244          0.3216  0.3248
SetB          0.4988  0.5071          0.3392  0.3430

**Bulgu (6):**
- Tohum sonucu biraz değiştiriyor (planın "neredeyse hiç değiştirmeyebilir" beklentisinden farklı, olduğu gibi yazıyoruz): 5 tohumda E0 ortalaması 0,418–0,424, bozuk 0,322–0,325; Set B ortalaması 0,499–0,507, bozuk 0,339–0,343.
- Tohum aralıkları dar (bozuk F1'de E0 için 0,003, Set B için 0,004) ve E0 ile Set B'nin bozuk aralıkları örtüşmüyor. E0 ile Set B arasındaki bozuk farkı (~0,015) bu 5 tohumda tohum oynamasından büyük.
- E9'un tohum aralığı ölçülmedi; "0,324 ile 0,340 arasındaki fark" sorusunun E9 kısmı Berat'ın bölümünde.

## 6. Deney 13: Çelişkili satırları atmadan
Soru: Eğitim katından çelişkili satırları (aynı özellikler, farklı etiket) atmak sonucu şişiriyor mu? `run_cv`'de kapatma seçeneği yok; katlar `get_folds()`'tan, döngü burada, metrikler `compute_metrics` ile. E0 ve Set B.

In [8]:
def run_without_conflict_drop(feats, feature_set_name, experiment):
    """Eğitim katından çelişkileri atmadan LightGBM eğitir, test katında ölçer. Satırı EXTRA_CSV'ye ekler."""
    y_true = data['Label'].to_numpy()
    oof = np.empty(len(data), dtype=object)
    fold_f1s = []
    for train_idx, test_idx in folds:
        train = data.iloc[train_idx]   # drop_conflicts bilerek çağrılmıyor
        model = LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)
        model.fit(train[feats], train['Label'])
        predictions = model.predict(data.iloc[test_idx][feats])
        oof[test_idx] = predictions
        fold_f1s.append(subset_macro_f1(y_true[test_idx], predictions, CLASSES))
    row = own_loop_row(experiment, feature_set_name, fold_f1s, oof.astype(y_true.dtype),
                       'EK-13 conflicting rows NOT dropped from training; own loop, not run_cv')
    print(experiment, '| kat F1:', np.round(fold_f1s, 4), '→', round(row['macro_f1_mean'], 4),
          '| bozuk F1:', round(row['macro_f1_broken'], 4))
    append_row(row, EXTRA_CSV)
    return row


conflict_rows = []
for config, (feats, feature_set_name) in SEED_CONFIGS.items():
    conflict_rows.append(results[config][0])
    row = run_without_conflict_drop(feats, feature_set_name, 'EK-13-' + config + '-keep-conflicts')
    results['13_' + config] = (row, None, None)
    conflict_rows.append(row)
display(pd.DataFrame(conflict_rows)[SHOW_COLS].round(4))

EK-13-E0-keep-conflicts | kat F1: [0.4309 0.5489 0.3174] → 0.4324 | bozuk F1: 0.3283
CSV'ye eklendi: ../reports/results_extra.csv


EK-13-SetB-keep-conflicts | kat F1: [0.4632 0.5591 0.4507] → 0.491 | bozuk F1: 0.3411
CSV'ye eklendi: ../reports/results_extra.csv


,experiment,model,feature_set,macro_f1_mean,macro_f1_std,macro_f1_clean,macro_f1_broken,recall_Bot,bot_ddos_confusion,note
0,EK-ref-E0,LGBMClassifier,F_TIMING,0.4183,0.1193,0.5123,0.3240,0.0308,0.1405,reference (same as E0)
1,EK-13-E0-keep-conflicts,LGBMClassifier,F_TIMING,0.4324,0.1157,0.5589,0.3283,0.0325,0.1381,EK-13 conflicting rows NOT dropped from traini...
2,EK-ref-SetB,LGBMClassifier,F_FULL,0.4988,0.0519,0.8365,0.3392,0.0334,0.1446,reference (same as Set B)
3,EK-13-SetB-keep-conflicts,LGBMClassifier,F_FULL,0.4910,0.0593,0.7765,0.3411,0.0356,0.1423,EK-13 conflicting rows NOT dropped from traini...


**Bulgu (13):**
- Çelişkileri atmadan: E0 0,432 ± 0,116 (resmî 0,418), bozuk 0,328 (0,324); Set B 0,491 ± 0,059 (resmî 0,499), bozuk 0,341 (0,339).
- Bozuk F1 iki yapılandırmada da en fazla 0,004 değişti; ortalama E0'da biraz yükseldi, Set B'de biraz düştü. Çelişkileri atmak bu ölçümde bozuk sonucu şişirmiyor.
- Değişimin çoğu sağlam alt kümede: E0 0,512 → 0,559, Set B 0,837 → 0,777. Yön iki yapılandırmada ters; bunu açıklayacak ek ölçüm yapmadık.

## 7. Deney 12: Oturum sınırı eşiği
Soru: Oturum neden 1 dakikalık boşlukla tanımlandı? Eşik 2 ve 5 dakika olunca oturum sayıları ve E0 ne oluyor?
`src/cv.py` değişmez: `make_groups`'un eşik parametreli bir kopyası burada yazılır; katlara dağıtım için `assign_sessions_to_folds` aynen kullanılır.

In [9]:
SESSION_GAPS = [SESSION_GAP_MINUTES, 2, 5]


def make_groups_with_gap(df, gap_minutes):
    """cv.make_groups ile aynı, sadece oturum eşiği parametre. Her satırın oturum grubunu döndürür."""
    groups = pd.Series(index=df.index, dtype=object)
    for label in df['Label'].unique():
        rows = df['Label'] == label
        minutes = np.sort(df.loc[rows, 'minute'].unique())
        gaps = np.diff(minutes, prepend=minutes[0])
        session_ids = np.cumsum(gaps > gap_minutes)
        minute_to_session = pd.Series(session_ids, index=minutes)
        groups[rows] = label + '_' + df.loc[rows, 'minute'].map(minute_to_session).astype(str)
    return groups


def folds_with_gap(df, gap_minutes):
    """get_folds ile aynı mantık, farklı oturum eşiğiyle. (folds, groups) döndürür."""
    groups = make_groups_with_gap(df, gap_minutes)
    fold_of_row = groups.map(assign_sessions_to_folds(df, groups)).to_numpy()
    gap_folds = []
    for fold_no in range(N_SPLITS):
        gap_folds.append((np.flatnonzero(fold_of_row != fold_no), np.flatnonzero(fold_of_row == fold_no)))
    return gap_folds, groups


session_counts = {}
for gap in SESSION_GAPS:
    gap_folds, groups = folds_with_gap(data, gap)
    session_counts[str(gap) + ' dk'] = groups.groupby(data['Label']).nunique()
    if gap == SESSION_GAP_MINUTES:
        continue   # 1 dk = resmî katlar, E0 referansı Bölüm 2'de
    print('=== EK-12 gap', gap, 'dk ===')
    print_fold_table(data, gap_folds)   # hangi katın testinde hangi sınıf var
    row, cm, oof = run_cv(make_lgbm(), data, F_TIMING, EXTRA_CSV, experiment='EK-12-E0-gap' + str(gap),
                          sampler='none', feature_set='F_TIMING', folds=gap_folds,
                          note='EK-12 session gap ' + str(gap) + ' min (official: 1 min)')
    results['12_gap' + str(gap)] = (row, cm, oof)

print('Sınıf başına oturum sayısı:')
display(pd.DataFrame(session_counts))
gap_rows = [results['E0'][0], results['12_gap2'][0], results['12_gap5'][0]]
display(pd.DataFrame(gap_rows)[SHOW_COLS].round(4))

=== EK-12 gap 2 dk ===
Test kısmındaki satır sayıları (sınıf × kat):
        fold_0  fold_1  fold_2
Label                         
Benign   76952   79965   76830
Bot          0    1757    7200
DDoS     23953   10599   15717
Dos      15589   15593   95324
Prob      2803    2316    2446
Test kısmındaki oturum sayıları (sınıf × kat):
        fold_0  fold_1  fold_2
Label                         
Benign      11       3       9
Bot          0       1       2
DDoS         1       1       2
Dos          4      17       2
Prob         1       1       4
Kat 0


  eğitim: 303447 satır (çelişki atılan: 4300 ) | test: 119297 | fit: 2.3 sn
Kat 1


  eğitim: 311119 satır (çelişki atılan: 5695 ) | test: 110230 | fit: 2.5 sn
Kat 2


  eğitim: 226414 satır (çelişki atılan: 3113 ) | test: 197517 | fit: 2.2 sn


Kat F1'leri: [0.5161 0.5546 0.2852] → 0.452 ± 0.1457
CSV'ye eklendi: ../reports/results_extra.csv


=== EK-12 gap 5 dk ===
Test kısmındaki satır sayıları (sınıf × kat):
        fold_0  fold_1  fold_2
Label                         
Benign   69853   73761   90133
Bot          0    1757    7200
DDoS     26316   23953       0
Dos      16976   14206   95324
Prob      3458    2316    1791
Test kısmındaki oturum sayıları (sınıf × kat):
        fold_0  fold_1  fold_2
Label                         
Benign      16       5       2
Bot          0       1       2
DDoS         3       1       0
Dos          5      16       2
Prob         2       1       3
Kat 0


... (3 outputs left out)

Kat F1'leri: [0.4011 0.5016 0.2456] → 0.3828 ± 0.129
CSV'ye eklendi: ../reports/results_extra.csv


Sınıf başına oturum sayısı:


,1 dk,2 dk,5 dk
Label,,,
Benign,23,18,13
Bot,3,2,2
DDoS,4,4,2
Dos,23,9,7
Prob,6,5,4


,experiment,model,feature_set,macro_f1_mean,macro_f1_std,macro_f1_clean,macro_f1_broken,recall_Bot,bot_ddos_confusion,note
0,EK-ref-E0,LGBMClassifier,F_TIMING,0.4183,0.1193,0.5123,0.3240,0.0308,0.1405,reference (same as E0)
1,EK-12-E0-gap2,LGBMClassifier,F_TIMING,0.4520,0.1457,0.5255,0.3087,0.0165,0.1511,EK-12 session gap 2 min (official: 1 min)
2,EK-12-E0-gap5,LGBMClassifier,F_TIMING,0.3828,0.1290,0.4853,0.2083,0.0156,0.2020,EK-12 session gap 5 min (official: 1 min)


**Bulgu (12):**
- Eşik büyüdükçe oturumlar birleşiyor: Bot 3 → 2 (2 dk ve 5 dk), DDoS 4 → 4 → 2, Dos 23 → 9 → 7, Benign 23 → 18 → 13.
- Bot'un 2 oturumu kaldığı için 2 ve 5 dakikada kat 0'ın testinde Bot yok (satır sayısı tablosu: 0 / 1.757 / 7.200); 5 dakikada kat 2'nin testinde DDoS da yok. Not: `print_fold_table`'ın oturum sayısı tablosu resmî 1 dakikalık oturumları sayıyor, bu deneyde satır sayısı tablosuna bakılmalı. Bu katlarda macro F1 o sınıf olmadan hesaplanıyor; bu yüzden E0 sonuçları (2 dk: 0,452 ± 0,146, bozuk 0,309; 5 dk: 0,383 ± 0,129, bozuk 0,208) resmî 0,418 ile doğrudan karşılaştırılamaz.
- Bu tablo, 1 dakikalık eşikle her sınıfın her katın testinde yer alabildiğini, daha büyük eşiklerde ise bunun sağlanamadığını gösteriyor.

## 8. Deney 15: Tekrarlanan satırlar
Soru: Veride birebir aynı satırlar var mı? `clean()` 5. adımda (FEATS + Label) aynı satırları zaten siliyor; yani resmî sonuçların hepsi tekrarsız veriyle. Burada sadece silinen tekrarları sınıf ve bozuk / sağlam ayrımıyla sayıyoruz (`drop_duplicates=False` ile). Yeni model eğitilmez.

In [10]:
data_with_duplicates = clean(pd.read_csv(DATA_PATH, low_memory=False), drop_duplicates=False)
is_duplicate = data_with_duplicates.duplicated(subset=FEATS + ['Label'])

duplicate_table = pd.DataFrame({
    'satır (tekrarlar dahil)': data_with_duplicates.groupby('Label').size(),
    'tekrar (silinen)': is_duplicate.groupby(data_with_duplicates['Label']).sum(),
    'tekrar, bozuk': (is_duplicate & data_with_duplicates['broken']).groupby(data_with_duplicates['Label']).sum(),
})
duplicate_table['tekrar oranı'] = (duplicate_table['tekrar (silinen)'] / duplicate_table['satır (tekrarlar dahil)']).round(3)
print('toplam tekrar:', is_duplicate.sum(), '| clean() sonrası satır:', len(data))
display(duplicate_table)

clean: dropped 70 rows with inf/NaN
clean: clipped negative values to 0 in 427 rows


clean: final shape (456974, 63)


toplam tekrar: 29930 | clean() sonrası satır: 427044


,satır (tekrarlar dahil),tekrar (silinen),"tekrar, bozuk",tekrar oranı
Label,,,,
Benign,260776,27029,787,0.104
Bot,9090,133,133,0.015
DDoS,51457,1188,1188,0.023
Dos,127766,1260,376,0.010
Prob,7885,320,320,0.041


**Bulgu (15):**
- `clean()` öncesi (inf/NaN sonrası) 456.974 satırın 29.930'u (FEATS + Label) birebir tekrar; `clean()` bunları siliyor ve 427.044 satır kalıyor. Resmî sonuçların hepsi bu tekrarsız veriyle.
- Tekrarların çoğu Benign'de (27.029; Benign'in %10,4'ü), büyük kısmı sağlam satırlarda (bozuk: 787). Bot, DDoS ve Prob'da oran %1,5–4,1; bu sınıfların bütün satırları bozuk olduğu için tekrarları da bozuk.
- Tekrarlar zaten silindiği için planlanan "sildikten sonra yeniden çalıştırma" resmî E0 / Set B ile aynı; ayrıca koşulmadı.

## 9. Deney 8: Oturum düzeyinde çoğunluk oyu
Soru: Akış yerine saldırı (oturum) düzeyinde bakınca ne oluyor? Yeni model eğitilmez: Set B'nin gruplu OOF tahminleri (Bölüm 2) kullanılır.
Her oturum, akışlarının en sık tahmin edilen sınıfıyla etiketlenir; aynısı 1 dakikalık pencereler (sınıf + dakika) için. Sınıf başına "doğru / toplam" sayı olarak verilir.
Sınırlılık: gerçek bir IDS oturum sınırlarını önceden bilmez; bu sadece toplu bakışın etkisine ilk kanıt.

In [11]:
def majority_vote_table(group_of_row, oof):
    """Her grup için çoğunluk tahminini gerçek sınıfla karşılaştırır. Sınıf başına doğru / toplam tablosu döndürür."""
    votes = pd.DataFrame({'group': group_of_row.to_numpy(), 'true': data['Label'].to_numpy(), 'pred': oof})
    rows = []
    for group_name, part in votes.groupby('group'):
        majority = part['pred'].value_counts().idxmax()
        rows.append({'Label': part['true'].iloc[0], 'correct': majority == part['true'].iloc[0]})
    per_group = pd.DataFrame(rows)
    table = per_group.groupby('Label')['correct'].agg(['sum', 'count'])
    table.columns = ['doğru', 'toplam']
    return table.reindex(CLASSES)


set_b_oof = results['SetB'][2].to_numpy()
session_of_row = make_groups(data)
window_of_row = data['Label'] + '_' + data['minute'].astype(str)

print('Oturum düzeyi (Set B):')
display(majority_vote_table(session_of_row, set_b_oof))
print('1 dakikalık pencere düzeyi (Set B):')
display(majority_vote_table(window_of_row, set_b_oof))

Oturum düzeyi (Set B):


,doğru,toplam
Label,,
Benign,19,23
Dos,4,23
DDoS,4,4
Bot,0,3
Prob,2,6


1 dakikalık pencere düzeyi (Set B):


,doğru,toplam
Label,,
Benign,572,653
Dos,25,118
DDoS,9,11
Bot,0,6
Prob,17,32


**Bulgu (8):**
- Oturum düzeyinde (Set B, çoğunluk oyu) doğru etiketlenen oturum: Benign 19/23, Dos 4/23, DDoS 4/4, Bot 0/3, Prob 2/6.
- 1 dakikalık pencere düzeyinde: Benign 572/653, Dos 25/118, DDoS 9/11, Bot 0/6, Prob 17/32.
- Toplu bakış DDoS'ta oturumların hepsini doğru etiketledi; Bot'ta hiçbir oturum ve pencere doğru etiketlenmedi (Bot akışlarının çoğu DDoS tahmin ediliyor). Dos'ta da çoğu oturum yanlış. Gerçek bir IDS oturum sınırlarını önceden bilmediği için bu sadece "Gelecek çalışma" için ilk kanıt.

## Devir notu (Doğukan → Berat, 28 Kasım 2026)
**Yapıldı:** referans (E0, Set B), deney 11 (tamamı), deney 14 (E0 + Set B), deney 6 (E0 + Set B), deney 13 (E0 + Set B), deney 12 (E0, 2 ve 5 dk), deney 15 (sayım; clean() tekrarları zaten siliyor), deney 8 (Set B). Notebook baştan sona ~5 dk.

**Kalan (plan sırasıyla):**
- 14 ve 6'ya **E9** eklenecek (E9 kodu `04_imbalance.ipynb`'de). 6 için `SEEDS` ve `SEED_CONFIGS` kullanılabilir.
- Öncelik 1: 4 → 10 → 3 → 1. Öncelik 2: 17 (12 ve 13 bitti). Öncelik 3: 9 → 7 → 5 → 2 → 16 → 18 (15 ve 8 bitti).
- Deney 3'te LR ve DT (F_FULL) zaten `reports/results_baseline.csv`'de (#20, aynı katlar); LR F_TIMING `results_formulation.csv`'de (D3-11.1). Tekrar koşmaya gerek yok, kalan 6 model yeterli.

**Kod notları:** `make_lgbm(seed)` sabit modeli kurar; `results` sözlüğü deney kodu → (row, cm, oof); `results_extra.csv` her Run All'da sıfırdan yazılır. `run_cv` yetmeyen deneyler için örnek: `run_shuffled_labels()` ve `run_without_conflict_drop()`, satır için `own_loop_row()`; farklı kat bölmesi için `folds_with_gap()` (katlar `get_folds`, metrikler `compute_metrics`, satır `run_cv` sütun sırasında).